# M02 – User-Defined Modules and Packages (PCAP 1.5)

**PCAP Alignment**: Section 1 (1.5). **Professional Focus**: Package layout, public API, naming.

---
## Learning Outcomes

- Create a **module** (single .py file) and import it; create a **package** (directory with **__init__.py**).

- Explain **__pycache__** and **__name__**; use **__name__** for script vs import.

- Distinguish **public** vs **private** (e.g. **_private**); use **__init__.py** and optional **__all__**.

- Use and search **nested packages**.

---
## Table of Contents

1. Modules and Packages (PCAP 1.5)
2. __pycache__ and __name__
3. __init__.py and Public API
4. Public vs Private Names
5. Nested Packages
6. Edge Cases and Pitfalls
7. More Examples
8. Practice

In [ ]:
print("__name__ =", __name__)
if __name__ == "__main__":
    print("Run as script")

---
## 1. Modules and Packages (PCAP 1.5)

**Modules** and **packages** are how Python organizes code for reuse and clear namespaces.

- **Module**: A single **.py** file. Any **.py** file can be imported as a module. Use **import mymod** to load it and access names as **mymod.foo**, or **from mymod import foo** to bring **foo** into the current scope. The file name (without .py) is the module name.

- **Package**: A **directory** that contains **__init__.py** (and usually other .py files or subdirectories). The directory name is the package name. Use **import pkg** to load the package (its **__init__.py** runs) or **from pkg import submod** to import a submodule. Packages let you group many modules under one name (e.g. **from pkg.utils import helper**).

- **Why use them:** Reuse code across scripts, avoid name clashes by keeping names in separate namespaces, improve maintainability and testability (each module can be tested or run on its own).

In [ ]:
# Example: __pycache__ — Python stores compiled bytecode for faster imports
import sys
print("When you import a module, Python creates .pyc files in __pycache__/")
print("Check if a module has cached bytecode:", hasattr(sys, "getallocatedblocks"))
# Tip: Delete __pycache__ folder to force recompilation; add to .gitignore

---
## 2. __pycache__ and __name__

**__pycache__**

- When you import a module, Python compiles the **.py** source to **bytecode** (a lower-level representation) and stores it in a **__pycache__** directory (usually as **.pyc** files). The next time you import the same module, Python can load from the cache if the source hasn't changed, which speeds up startup. You can delete **__pycache__** to force recompilation; it is often excluded from version control via **.gitignore**.

**__name__**

- Every module has a **__name__** attribute. When you **run a file as the main script** (e.g. **python mymod.py**), Python sets **__name__** to the string **"__main__"**. When the same file is **imported** by another module (e.g. **import mymod**), **__name__** is set to the **module name** (e.g. **"mymod"**). So you can write **if __name__ == "__main__":** and put code underneath that runs **only** when the file is executed directly — not when it is imported. This is the standard way to define tests, demos, or CLI entry points in a file that is also used as a library.

In [ ]:
# Run this cell in this notebook: __name__ is usually "__main__" in REPL/notebook
# When this same code is in a .py file: run as script -> "__main__"; import -> module name
print("__name__ =", __name__)
if __name__ == "__main__":
    print("Run as script")

---
## 3. __init__.py and Public API

**__init__.py**

- A directory is recognized as a **package** only if it contains a file named **__init__.py**. This file can be **empty** (just leave it blank) or it can run initialization code when the package is first imported. It is executed once per process when you first **import** the package or any of its submodules.

**Re-exporting a public API**

- Many packages use **__init__.py** to **re-export** a convenient set of names from submodules. For example, inside **pkg/__init__.py** you might write **from .submod import foo, bar**. Then users can write **from pkg import foo, bar** instead of **from pkg.submod import foo, bar**. This defines the package's "public API" in one place.

**__all__**

- **__all__** is an optional list of strings (e.g. **__all__ = ["foo", "bar"]**) that a module can define. When someone uses **from mod import ***, only the names listed in **__all__** are imported. Names starting with **_ ** are also not exported by **import ***. If **__all__** is not defined, **import *** brings in all names that do not start with **_ **.

In [ ]:
# Example: a built-in module's "public" names (those without leading underscore)
import sys
public = [n for n in dir(sys) if not n.startswith("_")]
print("Number of public names in sys:", len(public))
print("First 8:", public[:8])
# When you define __all__ in your module, "from mod import *" only brings in those names

In [ ]:
# Example: __all__ — control what "from mod import *" brings in
# In a module you could define:
# __all__ = ["public_func"]
# def public_func(): return "public"
# def _private_helper(): return "private"
# Then "from mod import *" only brings public_func
print("Define __all__ = ['foo', 'bar'] in your module to limit 'from mod import *'")

---
## 4. Public vs Private Names

Python does not enforce visibility at module level; the convention is:

- Names that **start with a single underscore** (e.g. **_helper**) are considered **private** or internal to the module. They are still visible and importable (e.g. **from mod import _helper** works), but tools and style guides treat them as "not part of the public API." When you use **from mod import ***, names starting with **_ ** are not exported; if you also define **__all__**, only names in **__all__** are exported.

- Names **without** a leading underscore are considered **public** and are the ones you expect users to use.

- **Double leading underscore** (e.g. **__name**) is used for **name mangling** in **classes** (see M05), not for module-level privacy. At module level, a single underscore is the convention.

---
## 5. Nested Packages

- Nested package: **pkg/subpkg/** with **__init__.py** in each directory. Import: **from pkg.subpkg import mod** or **from pkg.subpkg.mod import foo**.

- A directory **without** __init__.py is not a package (cannot import it as such).

In [ ]:
# Example: Single underscore — convention for "private" (still importable)
# In a module: def _helper(): return "internal"
# from mod import _helper  # works but discouraged
# from mod import *        # _helper is NOT imported
print("Names starting with _ are not exported by 'from mod import *'")

In [ ]:
# sys.path: where Python looks for modules. First entry is often script dir or cwd.
import sys
print("First 3 entries in sys.path:")
for i, p in enumerate(sys.path[:3]):
    print(f"  {i}: {p}")
# You can append a directory to allow importing from there (e.g. a local package)
# sys.path.append("/path/to/my/packages")

---
## 6. Edge Cases and Pitfalls

- **Circular imports**: A imports B, B imports A → can fail at load. Fix: move shared code to a third module or defer import inside a function.

- **__all__**: Define to control **from mod import ***; without it, all names not starting with _ are exported.

- Always use **if __name__ == "__main__":** for script-only code so the file can be imported without side effects.

---
## 7. More Examples

**Example: Simulating __name__ in a script**

In a file **mymod.py**, if you run **python mymod.py**, __name__ is **"__main__"**. If another file does **import mymod**, then in mymod, __name__ is **"mymod"**. So you can put tests or a CLI in **if __name__ == "__main__":** and they run only when the file is executed.

---
## 8. Practice

**Practice 1:** In a new file **mymath.py**, define **def double(x): return x * 2**. In another file or in a cell, **import mymath** and call **mymath.double(5)**. (If in notebook, ensure mymath.py is in sys.path or the same directory.)

In [ ]:
# Solution: mymath.py is in the same directory; import and call double(5)
import mymath
print("mymath.double(5) =", mymath.double(5))

**Practice 2:** Create a package directory **mypkg** with **__init__.py** that sets **VERSION = "1.0"**. Import and print **mypkg.VERSION**.

In [ ]:
# Solution: mypkg package with __init__.py that sets VERSION
import mypkg
print("mypkg.VERSION =", mypkg.VERSION)

**Practice 3:** In a module, add **__all__ = ["public_func"]**, define **def public_func(): pass** and **def _private_helper(): pass**. Try **from mod import *** and list **dir()** to confirm only **public_func** is in scope (when __all__ is used).

In [ ]:
# Solution: mod_for_all has __all__ = ["public_func"]
from mod_for_all import *
# Only names in __all__ are imported; _private_helper is not
print("public_func in dir():", "public_func" in dir())
print("_private_helper in dir():", "_private_helper" in dir())
print("public_func is callable:", callable(public_func))